# Experiment: quick CatBoost baseline on undersampled data

Rule filters + label-encoded numeric features, 100-iteration CatBoost evaluated with macro F1 on a stratified 20% split.

In [ ]:
DATA_DIR = "../data"

In [2]:
import time
import umap
import umap.plot
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
import keras.backend as K

from transformers import BertTokenizer, TFBertModel

from catboost import CatBoostClassifier

from nlp_id.lemmatizer import Lemmatizer
from nlp_id.tokenizer import Tokenizer, PhraseTokenizer
from nlp_id.postag import PosTag
from nlp_id.stopword import StopWord

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold
from sklearn import feature_selection as fs
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn import preprocessing
from sklearn.metrics.pairwise import cosine_similarity

from scipy import sparse

from sentence_transformers import SentenceTransformer

from IPython.display import IFrame
from IPython.core.display import display, HTML

/opt/conda/envs/tf-torch/lib/python3.10/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/opt/conda/envs/tf-torch/lib/python3.10/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/opt/conda/envs/tf-torch/lib/python3.10/site-packages/umap/distances.py:1086: NumbaDepreca

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train_sampled_data.csv")
train

,Unnamed: 0,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,...,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip,rule.mitre.tactic,hour,pairwise
0,0,38,2023-08-29 17:05:43.445,66.96.233.60,-,vm-director-nginx,30/Aug/2023:00:05:42 +0700,POST /lib/ajax/service.php?sesskey=FyapD11LyJ&...,200,98,...,0.03,10.45.184.157:32401,200,688312591,181,66.96.233.60,192.168.1.99,"[""Impact""]",17,0.126820
1,1,54,2023-08-29 17:08:23.939,118.99.125.4,-,vm-director-nginx,30/Aug/2023:00:08:23 +0700,POST /lib/ajax/service.php?sesskey=QY0HDC3szQ&...,200,1169,...,0.068,10.45.184.143:32401,200,688315946,158,118.99.125.4,192.168.1.99,"[""Impact""]",17,0.091207
2,2,98,2023-08-29 17:16:56.322,125.166.1.20,-,vm-director-nginx,30/Aug/2023:00:16:54 +0700,POST /lib/ajax/service.php?sesskey=9dbAEW9y6f&...,200,27,...,0.088,10.45.184.157:32401,200,688328194,209,125.166.1.20,192.168.1.99,"[""Impact""]",17,0.290300
3,3,99,2023-08-29 17:17:14.250,103.171.147.123,-,vm-director-nginx,30/Aug/2023:00:17:13 +0700,POST /lib/ajax/service.php?sesskey=nscrep2nd3&...,200,27,...,0.098,10.45.184.157:32401,200,688330190,90,103.171.147.123,192.168.1.99,"[""Impact""]",17,0.328679
4,4,187,2023-08-29 17:22:58.771,180.248.17.197,-,vm-director-nginx,30/Aug/2023:00:22:56 +0700,POST /lib/ajax/service.php?sesskey=ezddPBRNs5&...,200,6712,...,0.104,10.45.184.142:32401,200,688336653,219,180.248.17.197,192.168.1.99,"[""Impact""]",17,0.077081
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4753,4753,10060,2023-08-30 23:07:44.775,103.171.147.86,-,vm-director-nginx,31/Aug/2023:06:07:42 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,...,0.002,10.45.184.154:32442,404,694907441,12,103.171.147.86,192.168.1.99,"[""non-attack""]",23,0.983679
4754,4754,10187,2023-08-30 23:26:20.011,202.80.217.85,-,vm-director-nginx,31/Aug/2023:06:26:19 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,...,0.002,10.45.184.154:32442,404,694944826,2,202.80.217.85,192.168.1.99,"[""non-attack""]",23,0.983679
4755,4755,6735,2023-08-30 00:45:22.206,149.113.53.254,-,vm-director-nginx,30/Aug/2023:07:45:21 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,...,0.002,10.45.184.154:32442,404,689249474,4,149.113.53.254,192.168.1.99,"[""non-attack""]",0,0.983679
4756,4756,10177,2023-08-30 23:26:22.020,182.4.134.119,-,vm-director-nginx,31/Aug/2023:06:26:21 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,...,0.002,10.45.184.148:32442,404,694944919,2,182.4.134.119,192.168.1.99,"[""non-attack""]",23,0.983679


In [4]:
train.columns

Index(['Unnamed: 0', 'id', 'timestamp', 'remote_addr', 'remote_user',
       'hostname', 'time_local', 'request', 'status', 'body_bytes_sent',
       'http_referer', 'http_user_agent', 'http_x_forwarded_for',
       'time_iso8601', 'request_method', 'request_length', 'bytes_sent',
       'http_referer2', 'request_time', 'upstream_connect_time',
       'upstream_header_time', 'upstream_response_time', 'upstream_addr',
       'upstream_status', 'connection', 'connection_requests', 'data.srcip',
       'agent.ip', 'rule.mitre.tactic', 'hour', 'pairwise'],
      dtype='object')

In [6]:
#Time local
train['timestamp'] = pd.to_datetime(train['timestamp'], format="%Y-%m-%d %H:%M:%S.%f")

# Create a new column 'hour' containing the hour component
train['hour'] = train['timestamp'].dt.hour

In [7]:
train = train[~train['status'].isin([405, 407, 413, 422, 500, 502, 504])]

In [8]:
cross_tab = pd.crosstab(train['http_referer'], train['rule.mitre.tactic'])
cross_tab
# Filter rows where only the 'non attack' column has values greater than 0
cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
# Filter rows where only the 'non attack' column has values greater than 0
non_attack_http = cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
non_attack_http_list = non_attack_http.index.tolist()

for http_referer in non_attack_http_list:
    print(http_referer)


In [9]:
train = train[~train['http_referer'].isin(non_attack_http_list)]

In [10]:
train = train[~train['request_method'].isin(["HEAD"])]

In [11]:
train = train[~train['remote_user'].isin(["GapuraUb2020-Mobile", "google-oauth-client"])]

In [12]:
cross_tab = pd.crosstab(train['upstream_addr'], train['rule.mitre.tactic'])

# Filter rows where only the 'non attack' column has values greater than 0
cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
# Filter rows where only the 'non attack' column has values greater than 0
upstream_addr_nonattack = cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
upstream_addr_list = upstream_addr_nonattack.index.tolist()
upstream_addr_list

[]

In [73]:
train = train[~train['upstream_addr'].isin(upstream_addr_list)]

In [15]:
# Drop useless column
train_drop = train.drop(['Unnamed: 0','timestamp', 'remote_addr', 'remote_user', 'hostname', 'time_local', 'request', 'http_referer', 'http_user_agent', 'http_x_forwarded_for', 'time_iso8601', 'http_referer2', 'upstream_addr', 'upstream_status', 'connection', 'data.srcip', 'agent.ip','rule.mitre.tactic'], axis=1)

In [16]:
train_drop.columns

Index(['id', 'status', 'body_bytes_sent', 'request_method', 'request_length',
       'bytes_sent', 'request_time', 'upstream_connect_time',
       'upstream_header_time', 'upstream_response_time', 'connection_requests',
       'hour', 'pairwise'],
      dtype='object')

In [17]:
train_drop.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4758 entries, 0 to 4757
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      4758 non-null   int64  
 1   status                  4758 non-null   int64  
 2   body_bytes_sent         4758 non-null   int64  
 3   request_method          4758 non-null   object 
 4   request_length          4758 non-null   int64  
 5   bytes_sent              4758 non-null   int64  
 6   request_time            4758 non-null   float64
 7   upstream_connect_time   4758 non-null   object 
 8   upstream_header_time    4758 non-null   object 
 9   upstream_response_time  4758 non-null   object 
 10  connection_requests     4758 non-null   int64  
 11  hour                    4758 non-null   int64  
 12  pairwise                4758 non-null   float64
dtypes: float64(2), int64(7), object(4)
memory usage: 520.4+ KB


In [18]:
from sklearn.preprocessing import LabelEncoder
# Create a LabelEncoder instance
encoder = LabelEncoder()

# Fit and transform the 'request_method' column
train_drop['request_method'] = encoder.fit_transform(train_drop['request_method'])


In [19]:
train_drop.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4758 entries, 0 to 4757
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      4758 non-null   int64  
 1   status                  4758 non-null   int64  
 2   body_bytes_sent         4758 non-null   int64  
 3   request_method          4758 non-null   int64  
 4   request_length          4758 non-null   int64  
 5   bytes_sent              4758 non-null   int64  
 6   request_time            4758 non-null   float64
 7   upstream_connect_time   4758 non-null   object 
 8   upstream_header_time    4758 non-null   object 
 9   upstream_response_time  4758 non-null   object 
 10  connection_requests     4758 non-null   int64  
 11  hour                    4758 non-null   int64  
 12  pairwise                4758 non-null   float64
dtypes: float64(2), int64(8), object(3)
memory usage: 520.4+ KB


In [20]:
# Loop through each column
for column in train_drop.columns:
    train_drop[column] = pd.to_numeric(train_drop[column], errors='coerce')

# 'coerce' option will convert non-numeric values to NaN if they can't be converted
# If you want to drop rows with NaN values, you can use df.dropna()

print(train_drop.dtypes)

id                          int64
status                      int64
body_bytes_sent             int64
request_method              int64
request_length              int64
bytes_sent                  int64
request_time              float64
upstream_connect_time     float64
upstream_header_time      float64
upstream_response_time    float64
connection_requests         int64
hour                        int64
pairwise                  float64
dtype: object


In [21]:
import pandas as pd

# Assuming you have your DataFrame already loaded, if not, load your data here

# Calculate the mean for each numeric column
mean_values = train_drop.mean()

# Fill missing values with the mean for each column
train_drop.fillna(mean_values, inplace=True)

# Display the DataFrame with missing values filled with means
train_drop.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4758 entries, 0 to 4757
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      4758 non-null   int64  
 1   status                  4758 non-null   int64  
 2   body_bytes_sent         4758 non-null   int64  
 3   request_method          4758 non-null   int64  
 4   request_length          4758 non-null   int64  
 5   bytes_sent              4758 non-null   int64  
 6   request_time            4758 non-null   float64
 7   upstream_connect_time   4758 non-null   float64
 8   upstream_header_time    4758 non-null   float64
 9   upstream_response_time  4758 non-null   float64
 10  connection_requests     4758 non-null   int64  
 11  hour                    4758 non-null   int64  
 12  pairwise                4758 non-null   float64
dtypes: float64(5), int64(8)
memory usage: 520.4 KB


In [31]:
from sklearn.metrics import accuracy_score

In [45]:
train = train[train['rule.mitre.tactic'] != '["Credential Access"]']

In [47]:
train_drop = train_drop.loc[train.index]

In [46]:
train['rule.mitre.tactic'].value_counts()

["non-attack"]         2379
["Impact"]             1784
["Reconnaissance"]      561
["Defense Evasion"]      29
["Initial Access"]        4
Name: rule.mitre.tactic, dtype: int64

In [49]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(train_drop.drop(['id','pairwise'],axis=1), train['rule.mitre.tactic'], test_size=0.2, random_state=42, stratify=train['rule.mitre.tactic'])

# Step 4: Train a CatBoost classifier
# You can customize the hyperparameters based on your dataset and requirements
catboost_classifier = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, verbose=100)

# Fit the classifier to the training data
catboost_classifier.fit(X_train, y_train)

# Step 5: Use the trained classifier to make predictions
y_pred = catboost_classifier.predict(X_test)

# Evaluate the model's F1 score
f1 = f1_score(y_test, y_pred, average='macro')
print(f'F1 Score: {f1 * 100:.2f}%')
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

0:	learn: 1.2813047	total: 31.9ms	remaining: 3.16s
99:	learn: 0.0372966	total: 2.22s	remaining: 0us
F1 Score: 95.12%
Accuracy: 99.05%


In [ ]:
test = pd.read_csv(f"{DATA_DIR}/train_sampled_data.csv")
train